In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler,MinMaxScaler,LabelEncoder,OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix,f1_score,classification_report,accuracy_score,recall_score,precision_score
from sklearn.pipeline import Pipeline
import pickle
from sklearn.ensemble import RandomForestClassifier,BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

In [3]:
# Load dataset
data = pd.read_csv("matches.csv")
data.sample(3)

,id,season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
642,7900,2018,Hyderabad,12/04/18,Mumbai Indians,Sunrisers Hyderabad,Sunrisers Hyderabad,field,normal,0,Sunrisers Hyderabad,0,1,Rashid Khan,"Rajiv Gandhi International Stadium, Uppal",O Nandan,Nigel Llong,Vineet Kulkarni
449,450,2013,Dharamsala,2013-05-18,Kings XI Punjab,Mumbai Indians,Mumbai Indians,field,normal,0,Kings XI Punjab,50,0,Azhar Mahmood,Himachal Pradesh Cricket Association Stadium,HDPK Dharmasena,CK Nandan,NaN
16,17,2017,Bangalore,2017-04-16,Rising Pune Supergiant,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Rising Pune Supergiant,27,0,BA Stokes,M Chinnaswamy Stadium,KN Ananthapadmanabhan,C Shamshuddin,NaN


In [4]:
# EXtract importance col form the dataset
data = data[['team1', 'team2', 'toss_winner', 'toss_decision', 'venue', 'city',"winner"]]

In [5]:
data.sample(5)

,team1,team2,toss_winner,toss_decision,venue,city,winner
533,Mumbai Indians,Royal Challengers Bangalore,Royal Challengers Bangalore,field,M Chinnaswamy Stadium,Bangalore,Mumbai Indians
342,Kings XI Punjab,Chennai Super Kings,Kings XI Punjab,bat,"MA Chidambaram Stadium, Chepauk",Chennai,Kings XI Punjab
682,Mumbai Indians,Rajasthan Royals,Rajasthan Royals,field,Wankhede Stadium,Mumbai,Rajasthan Royals
163,Kings XI Punjab,Deccan Chargers,Deccan Chargers,field,New Wanderers Stadium,Johannesburg,Kings XI Punjab
173,Deccan Chargers,Royal Challengers Bangalore,Royal Challengers Bangalore,field,New Wanderers Stadium,Johannesburg,Deccan Chargers


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 756 entries, 0 to 755
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   team1          756 non-null    object
 1   team2          756 non-null    object
 2   toss_winner    756 non-null    object
 3   toss_decision  756 non-null    object
 4   venue          756 non-null    object
 5   city           749 non-null    object
 6   winner         752 non-null    object
dtypes: object(7)
memory usage: 41.5+ KB


In [7]:
data.isnull().mean() * 100

team1            0.000000
team2            0.000000
toss_winner      0.000000
toss_decision    0.000000
venue            0.000000
city             0.925926
winner           0.529101
dtype: float64

In [8]:
from sklearn.impute import SimpleImputer
# filling missing values
si = SimpleImputer(strategy="most_frequent")
data[["city"]] = si.fit_transform(data[["city"]])

In [9]:
data.isnull().mean() * 100

team1            0.000000
team2            0.000000
toss_winner      0.000000
toss_decision    0.000000
venue            0.000000
city             0.000000
winner           0.529101
dtype: float64

In [10]:
data.duplicated().sum()

142

In [11]:
# Removing the duplicate
data.drop_duplicates(inplace=True)

In [12]:
data.duplicated().sum()

0

In [13]:
print("Original Winner Distribution:")
print(data["winner"].value_counts())

Original Winner Distribution:
winner
Mumbai Indians                 81
Chennai Super Kings            80
Royal Challengers Bangalore    70
Kings XI Punjab                68
Rajasthan Royals               64
Kolkata Knight Riders          61
Delhi Daredevils               58
Sunrisers Hyderabad            48
Deccan Chargers                27
Pune Warriors                  12
Gujarat Lions                  11
Delhi Capitals                 10
Rising Pune Supergiant          9
Kochi Tuskers Kerala            6
Rising Pune Supergiants         5
Name: count, dtype: int64


In [14]:
# Handling the imblance class using sampling techniques
# get the largets winner size
max_size = data["winner"].value_counts().max()

# preform oversampling

balanced_data = data.groupby('winner').apply(lambda x : x.sample(max_size,replace = True)).reset_index(drop=True)
 # shuffle the dataset to avoid any order bias
data = balanced_data.sample(frac=1).reset_index(drop=True)

# check the balance winnner distribution
print("/n Balanced Wineer Distribution (after oversampling):")
print(data["winner"].value_counts())

/n Balanced Wineer Distribution (after oversampling):
winner
Sunrisers Hyderabad            81
Gujarat Lions                  81
Chennai Super Kings            81
Rising Pune Supergiant         81
Kolkata Knight Riders          81
Kochi Tuskers Kerala           81
Pune Warriors                  81
Delhi Capitals                 81
Kings XI Punjab                81
Royal Challengers Bangalore    81
Rising Pune Supergiants        81
Deccan Chargers                81
Mumbai Indians                 81
Rajasthan Royals               81
Delhi Daredevils               81
Name: count, dtype: int64


In [15]:
data.sample(5)

,team1,team2,toss_winner,toss_decision,venue,city,winner
658,Kings XI Punjab,Pune Warriors,Kings XI Punjab,bat,"Punjab Cricket Association Stadium, Mohali",Chandigarh,Pune Warriors
1171,Mumbai Indians,Deccan Chargers,Deccan Chargers,field,Dr DY Patil Sports Academy,Mumbai,Deccan Chargers
893,Pune Warriors,Kolkata Knight Riders,Kolkata Knight Riders,field,JSCA International Stadium Complex,Ranchi,Pune Warriors
201,Kolkata Knight Riders,Mumbai Indians,Kolkata Knight Riders,bat,Sheikh Zayed Stadium,Abu Dhabi,Kolkata Knight Riders
363,Mumbai Indians,Kings XI Punjab,Kings XI Punjab,field,IS Bindra Stadium,Mohali,Kings XI Punjab


In [16]:
data["toss_decision"].value_counts()

toss_decision
field    788
bat      427
Name: count, dtype: int64

In [17]:
x = data.iloc[:,:-1]
y = data[["winner"]]

In [18]:
import string
pu = string.punctuation
pu

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [19]:

# preprocessing
def preprocessing(text):
    text = text.lower()
    text = text.replace(" ","")
    return text.translate(str.maketrans("","",pu))

In [20]:
x["team1"] = x["team1"].apply(preprocessing)
x["team2"] = x["team2"].apply(preprocessing)
x["toss_winner"] = x["toss_winner"].apply(preprocessing)
x["city"] = x["city"].apply(preprocessing)
x["venue"] = x["venue"].apply(preprocessing)
y["winner"] = y["winner"].apply(preprocessing)

In [21]:
x.sample(10)

,team1,team2,toss_winner,toss_decision,venue,city
454,deccanchargers,mumbaiindians,deccanchargers,bat,supersportpark,centurion
509,chennaisuperkings,kochituskerskerala,kochituskerskerala,field,nehrustadium,kochi
631,deccanchargers,royalchallengersbangalore,royalchallengersbangalore,field,vidarbhacricketassociationstadiumjamtha,nagpur
193,kingsxipunjab,mumbaiindians,mumbaiindians,field,punjabcricketassociationstadiummohali,chandigarh
1028,kingsxipunjab,delhicapitals,delhicapitals,field,isbindrastadium,mohali
1067,sunrisershyderabad,royalchallengersbangalore,sunrisershyderabad,bat,rajivgandhiinternationalstadiumuppal,hyderabad
704,mumbaiindians,gujaratlions,gujaratlions,field,greenpark,kanpur
144,rajasthanroyals,kolkataknightriders,rajasthanroyals,bat,edengardens,kolkata
1019,kingsxipunjab,risingpunesupergiants,kingsxipunjab,bat,drysrajasekharareddyacavdcacricketstadium,visakhapatnam
1192,chennaisuperkings,kingsxipunjab,chennaisuperkings,bat,sheikhzayedstadium,abudhabi


In [22]:
le = LabelEncoder()
y_encoder = le.fit_transform(y)

In [23]:
# spliting the data set
x_train,x_test,y_train,y_test = train_test_split(x,y_encoder,test_size=0.2,random_state=42)

In [24]:
print(f"x_train shape : {x_train.shape} {"\t"} x_test shape : {x_test.shape} {"\n"}y_trian shape : {y_train.shape} {"\t"} y_test shape : {y_test.shape}")

x_train shape : (972, 6) 	 x_test shape : (243, 6) 
y_trian shape : (972,) 	 y_test shape : (243,)


In [25]:
trf = ColumnTransformer(transformers=[
    ("OHE",OneHotEncoder(drop="first",sparse_output=True),["team1","team2","toss_winner","toss_decision","venue","city"])
],remainder="passthrough")

x_train_trf = trf.fit_transform(x_train)
x_test_trf = trf.transform(x_test)

In [26]:
x_train_trf.shape

(972, 113)

In [27]:
x_test_trf.shape

(243, 113)

In [28]:
def tranning(model):
    model1 = model
    model1.fit(x_train_trf, y_train)

    y_pred = model1.predict(x_test_trf)

    print(f"accuracy score : {accuracy_score(y_test, y_pred)}")
    print(f"F1 score : {f1_score(y_test, y_pred, average='macro')}")
    print(f"recall score : {recall_score(y_test, y_pred, average='macro')}")
    print(f"precision score : {precision_score(y_test, y_pred, average='macro')}")
    print("confusion_matrix :\n", confusion_matrix(y_test, y_pred))
    print("classification_report :\n", classification_report(y_test, y_pred))

    return model1

In [29]:
print("LogisticRegression Model")
logist_r = tranning(LogisticRegression())
print(logist_r)

LogisticRegression Model
accuracy score : 0.8065843621399177
F1 score : 0.8139811414992905
recall score : 0.8213473635068061
precision score : 0.8135204842170787
confusion_matrix :
 [[ 9  0  0  0  0  2  0  1  2  0  0  0  0  0  1]
 [ 0 14  0  3  0  0  0  1  0  0  1  0  0  0  0]
 [ 0  0 12  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 2  0  0 11  1  1  0  1  1  1  2  0  0  1  0]
 [ 0  0  0  0 15  0  0  0  0  0  0  0  0  0  0]
 [ 1  1  0  0  1 12  0  2  0  0  0  0  0  1  0]
 [ 0  0  0  0  0  0 13  0  0  0  0  0  0  0  0]
 [ 0  0  0  2  0  0  0 12  1  0  0  0  0  2  1]
 [ 0  0  0  0  0  1  0  1 11  0  1  0  0  2  0]
 [ 0  0  0  0  0  0  0  0  0 18  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 11  0  0  0  1]
 [ 0  0  0  0  0  0  0  0  0  0  0 20  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0 13  0  0]
 [ 2  0  0  0  0  0  0  1  2  0  0  0  0 11  1]
 [ 0  0  0  0  0  0  0  1  0  0  1  0  0  0 14]]
classification_report :
               precision    recall  f1-score   support

           0     

In [30]:
print("RandomForestClassifier model")
randomforest_c = tranning(RandomForestClassifier())
print(randomforest_c)

RandomForestClassifier model
accuracy score : 0.8436213991769548
F1 score : 0.8442966422378188
recall score : 0.8516705571117336
precision score : 0.8434842453727904
confusion_matrix :
 [[11  0  0  0  0  1  0  1  1  0  0  0  0  0  1]
 [ 0 19  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 12  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0 16  1  1  0  0  1  0  1  0  0  1  0]
 [ 0  0  0  0 15  0  0  0  0  0  0  0  0  0  0]
 [ 2  1  0  0  1 13  0  1  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 13  0  0  0  0  0  0  0  0]
 [ 1  0  0  3  0  0  0 12  1  0  0  0  0  1  0]
 [ 0  0  0  0  0  1  0  1 11  0  1  0  0  2  0]
 [ 0  0  0  0  0  0  0  0  0 18  0  0  0  0  0]
 [ 1  0  0  0  0  0  0  0  0  0 11  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 20  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0 13  0  0]
 [ 4  0  0  0  0  0  0  1  2  1  0  0  0  7  2]
 [ 0  0  0  0  0  0  0  1  0  0  1  0  0  0 14]]
classification_report :
               precision    recall  f1-score   support

           0 

In [31]:
print("XGBClassifier model")
xgb_c = tranning(XGBClassifier())
print(xgb_c)

XGBClassifier model
accuracy score : 0.8600823045267489
F1 score : 0.8616263071268232
recall score : 0.8681310301898537
precision score : 0.8602287581699346
confusion_matrix :
 [[11  0  0  0  0  1  0  1  1  0  0  0  0  0  1]
 [ 0 19  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 12  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0 15  1  1  0  1  1  0  1  0  0  1  0]
 [ 0  0  0  0 15  0  0  0  0  0  0  0  0  0  0]
 [ 2  0  0  0  1 14  0  1  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 13  0  0  0  0  0  0  0  0]
 [ 0  0  0  1  0  0  0 12  1  0  1  0  0  2  1]
 [ 0  0  0  0  0  1  0  1 11  0  1  0  0  2  0]
 [ 0  0  0  0  0  0  0  0  0 18  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  1 11  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 20  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0 13  0  0]
 [ 2  0  0  0  0  0  0  1  2  1  0  0  0 10  1]
 [ 0  0  0  0  0  0  0  0  0  0  1  0  0  0 15]]
classification_report :
               precision    recall  f1-score   support

           0       0.7

In [32]:
print("BaggingClassifier Model")
bagging_c = tranning(BaggingClassifier(estimator=DecisionTreeClassifier(),n_estimators=100,random_state=42))
print(bagging_c)

BaggingClassifier Model
accuracy score : 0.8600823045267489
F1 score : 0.8591286121286122
recall score : 0.8664192343604108
precision score : 0.8581677584541362
confusion_matrix :
 [[11  0  0  0  0  1  0  1  1  0  0  0  0  0  1]
 [ 0 19  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 12  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0 17  1  1  0  0  1  0  0  0  0  1  0]
 [ 0  0  0  0 15  0  0  0  0  0  0  0  0  0  0]
 [ 1  1  0  0  1 14  0  1  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 13  0  0  0  0  0  0  0  0]
 [ 1  0  0  1  0  0  0 13  1  0  0  0  0  1  1]
 [ 0  0  0  0  0  1  0  1 11  0  1  0  0  2  0]
 [ 0  0  0  0  0  0  0  0  0 18  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  1 11  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 20  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0 13  0  0]
 [ 2  0  0  0  0  0  0  2  2  2  0  0  0  7  2]
 [ 0  0  0  0  0  0  0  0  0  0  1  0  0  0 15]]
classification_report :
               precision    recall  f1-score   support

           0      

In [33]:
from sklearn.svm import SVC
print("SVC MOdel")
svc = tranning(SVC(kernel='rbf', C=1.0))
print(svc)

SVC MOdel
accuracy score : 0.8189300411522634
F1 score : 0.8239663097996488
recall score : 0.8325887021475257
precision score : 0.8238684191625368
confusion_matrix :
 [[ 9  0  0  0  0  1  0  1  2  0  0  0  0  1  1]
 [ 0 19  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 12  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 2  0  0 12  1  1  0  1  1  0  2  0  0  1  0]
 [ 0  0  0  0 15  0  0  0  0  0  0  0  0  0  0]
 [ 1  2  0  0  1 10  0  3  0  0  0  0  0  1  0]
 [ 0  0  0  0  0  0 13  0  0  0  0  0  0  0  0]
 [ 0  0  0  3  0  0  0 11  1  0  0  0  0  2  1]
 [ 0  0  0  0  0  1  0  1 11  0  1  0  0  2  0]
 [ 0  0  0  0  0  0  0  0  0 18  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 12  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 20  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0 13  0  0]
 [ 2  0  0  0  0  0  0  2  2  0  0  0  0 10  1]
 [ 0  0  0  0  0  0  0  1  0  0  1  0  0  0 14]]
classification_report :
               precision    recall  f1-score   support

           0       0.64      0.6

In [34]:
print("DecisionTreeClassifier MOdel")
decisiontree_c = tranning(DecisionTreeClassifier(max_depth=6,
    min_samples_split=10,
    min_samples_leaf=5,
    criterion="gini",
    random_state=42))
print(decisiontree_c)

DecisionTreeClassifier MOdel
accuracy score : 0.3004115226337449
F1 score : 0.29667157958129287
recall score : 0.3390598290598291
precision score : 0.31470361723170714
confusion_matrix :
 [[ 0  0  1  0  0  0  0  0  0  0 14  0  0  0  0]
 [ 0  0  0  0  0  0  1  0  0  0 18  0  0  0  0]
 [ 0  0  7  0  0  0  0  0  0  0  5  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 21  0  0  0  0]
 [ 0  0  0  0 11  0  0  0  0  0  4  0  0  0  0]
 [ 0  0  0  0  1  0  0  0  0  0 17  0  0  0  0]
 [ 0  0  0  0  0  0 10  0  0  0  3  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 17  1  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 16  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 18  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 12  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 20  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0 13  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 17  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 16  0  0  0  0]]
classification_report :
               precision    recall  f1-score   support

           

In [35]:
# Comparing to all other model but XGBoostClassifier perform well:
with open("x_g_b_model.pkl", "wb") as file:
    pickle.dump(xgb_c, file)

In [36]:
new_data = pd.DataFrame([{
    "team1": "mumbaiindians",
    "team2": "chennai  superkings",
    "toss_winner": "mumbai indians",
    "toss_decision" : "field",
    "venue": "wankhedestadium",
    "city": "mumbai"
}])

In [37]:
x

,team1,team2,toss_winner,toss_decision,venue,city
0,sunrisershyderabad,royalchallengersbangalore,royalchallengersbangalore,field,rajivgandhiintlcricketstadium,hyderabad
1,kolkataknightriders,gujaratlions,gujaratlions,field,edengardens,kolkata
2,chennaisuperkings,mumbaiindians,chennaisuperkings,bat,ferozshahkotla,delhi
3,risingpunesupergiant,royalchallengersbangalore,royalchallengersbangalore,field,maharashtracricketassociationstadium,pune
4,kolkataknightriders,delhidaredevils,kolkataknightriders,bat,edengardens,kolkata
...,...,...,...,...,...,...
1210,gujaratlions,sunrisershyderabad,sunrisershyderabad,field,ferozshahkotla,delhi
1211,kolkataknightriders,mumbaiindians,mumbaiindians,field,edengardens,kolkata
1212,mumbaiindians,risingpunesupergiants,mumbaiindians,bat,wankhedestadium,mumbai
1213,delhidaredevils,rajasthanroyals,rajasthanroyals,field,ferozshahkotla,delhi


In [38]:
new_data["team1"] = new_data["team1"].apply(preprocessing)
new_data["team2"] = new_data["team2"].apply(preprocessing)
new_data["toss_winner"] = new_data["toss_winner"].apply(preprocessing)
new_data["city"] = new_data["city"].apply(preprocessing)
new_data["venue"] =new_data["venue"].apply(preprocessing)

In [39]:
new_data_trf = trf.transform(new_data)

In [40]:
xgb_c.predict(x_test_trf)[0]

8

In [42]:
print(le.inverse_transform(xgb_c.predict(x_test_trf))[0])

mumbaiindians
